# Customer Churn: Responsible Baseline Notebook

Educational demonstration only. The dataset has 12 rows, so test metrics are unstable and must not be interpreted as production performance.

## 1. Load data and run quality checks

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             balanced_accuracy_score, accuracy_score)

DATA_PATH = 'customer-churn-training.csv'  # Keep the CSV beside this notebook
df = pd.read_csv(DATA_PATH)
display(df.head())
print('Shape:', df.shape)
print('Missing values:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate customer IDs:', df['customer_id'].duplicated().sum())
print('Target counts:\n', df['churned'].value_counts(dropna=False))


## 2. Define target and exclude the identifier

`customer_id` is an identifier, not a useful feature. Preprocessing is fitted within each training pipeline to reduce leakage.

In [ ]:
target = 'churned'
X = df.drop(columns=['customer_id', target])
y = df[target].astype(int)

numeric_features = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
categorical_features = ['plan_type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)
print('Train rows:', len(X_train), '| Test rows:', len(X_test))
print('Train target counts:\n', y_train.value_counts())
print('Test target counts:\n', y_test.value_counts())


## 3. Compare majority-class baseline with logistic regression

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocess = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_features),
    ('categorical', categorical_pipe, categorical_features)
])

models = {
    'Majority-class baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': Pipeline([
        ('preprocess', preprocess),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision_churn': precision_score(y_test, pred, zero_division=0),
        'recall_churn': recall_score(y_test, pred, zero_division=0),
        'f1_churn': f1_score(y_test, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred)
    })
    print('\n---', name, '---')
    print('Confusion matrix [ [TN, FP], [FN, TP] ]:\n', confusion_matrix(y_test, pred, labels=[0, 1]))
    print(classification_report(y_test, pred, labels=[0, 1], zero_division=0))

display(pd.DataFrame(results))


## 4. Interpretation and limitations

- Compare the model with the majority baseline; do not assume the more complex model is better.
- False negatives are churners missed by the model; false positives are customers flagged unnecessarily.
- A tiny holdout can change metrics dramatically with just one prediction. Do not report these values as evidence of production readiness.
- Confirm data provenance, label definition, feature time windows, and prediction cutoff before any real use.
- For production research, obtain a larger time-stamped dataset, use temporal validation, assess calibration and fairness where feasible, and maintain human review plus a non-ML fallback.

## 5. Proposed fallback

If the model or pipeline fails, continue the ordinary customer-support process. A simple, documented rule can be evaluated as a benchmark, but it must not be treated as validated merely because it is easy to explain.
